# Pipeline Bronze para Silver — CineData Analytics

Este notebook implementa o pipeline de conformação, enriquecimento e higienização dos dados da camada **Bronze** para a camada **Silver**.

### Diretrizes de Engenharia e Padrões Aplicados:
- **Contratos Canônicos de Esquema**: Validação estrita de tipos e estrutura via `StructType` para cada tabela de destino.
- **Tradução e Nomenclatura**: Padronização dos nomes de campos em português (`snake_case`).
- **Rastreabilidade**: Preservação da linhagem temporal mantendo o `ingestion_datetime` gerado na ingestão Bronze.
- **Modularidade e Coesão**: Funções utilitárias puras e constantes de domínio aplicadas diretamente em cada etapa.

### Premissa de Arquitetura e Duplicação de código dos Notebooks:
- Funções utilitárias (como persistência, validação e qualidade) foram mantidas diretamente em cada notebook. Essa duplicação visa dispensar módulos compartilhados ou abstrações externas no Databricks.


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

from pyspark.sql import Column, DataFrame, Row, SparkSession, Window
from pyspark.sql.functions import (
    coalesce,
    col,
    create_map,
    explode,
    expr,
    initcap,
    last,
    length,
    lit,
    regexp_extract,
    regexp_replace,
    row_number,
    split,
    to_date,
    trim,
    try_to_date,
    upper,
    when,
    year,
)
from pyspark.sql.functions import max as spark_max
from pyspark.sql.functions import min as spark_min
from pyspark.sql.types import (
    DateType,
    DecimalType,
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.ansi.enabled", "false")

spark.sql("CREATE DATABASE IF NOT EXISTS silver")

DEFAULT_SENTINEL_VALUES = [
    "UNKNOWN", "NÃO INFORMADO", "NAO INFORMADO", "NONE", "N/A", "NULL", ""
]

def enforce_dataframe_schema(
    dataframe: DataFrame,
    expected_schema: StructType,
    strict_columns: bool = True
) -> DataFrame:
    """Aplica tipos e ordenação estrita ao DataFrame conforme o esquema StructType canônico definido."""
    actual_column_names = set(dataframe.columns)
    expected_column_names = [field.name for field in expected_schema.fields]
    missing_columns = set(expected_column_names) - actual_column_names

    if missing_columns:
        raise ValueError(f"Colunas obrigatórias ausentes no DataFrame: {missing_columns}")

    if strict_columns:
        unexpected_columns = actual_column_names - set(expected_column_names)
        if unexpected_columns:
            raise ValueError(f"Colunas imprevistas encontradas no DataFrame: {unexpected_columns}")

    ordered_column_expressions = [
        col(field.name).cast(field.dataType).alias(field.name)
        for field in expected_schema.fields
    ]
    return dataframe.select(ordered_column_expressions)

def write_dataframe(
    dataframe: DataFrame,
    table_name: str,
    save_mode: str = "overwrite"
) -> None:
    """Persiste o DataFrame como tabela Delta no catálogo especificado."""
    (
        dataframe.write
        .format("delta")
        .mode(save_mode)
        .saveAsTable(table_name)
    )

def deduplicate_latest(
    dataframe: DataFrame,
    business_key_columns: list[str] | str | None = None,
    business_key_column: str | None = None,
    timestamp_column: str = "ingestion_datetime"
) -> DataFrame:
    """Remove duplicatas mantendo o registro mais recente baseado no timestamp de ingestão."""
    target_key = business_key_column if business_key_column is not None else business_key_columns
    if target_key is None:
        raise ValueError("Chave de negócio obrigatória para deduplicação.")
    key_columns_list = [target_key] if isinstance(target_key, str) else target_key
    window_by_business_key = (
        Window
        .partitionBy(*key_columns_list)
        .orderBy(col(timestamp_column).desc())
    )
    return (
        dataframe
        .withColumn("_row_order_number", row_number().over(window_by_business_key))
        .filter(col("_row_order_number") == 1)
        .drop("_row_order_number")
    )

def sanitize_string(column: Column, sentinel_values: list[str] | None = None) -> Column:
    """Remove espaços em branco nas extremidades e converte valores sentinelas para NULL."""
    active_sentinels = sentinel_values if sentinel_values is not None else DEFAULT_SENTINEL_VALUES
    uppercase_sentinels = [single_sentinel.upper() for single_sentinel in active_sentinels]
    trimmed_column = trim(column)
    return when(upper(trimmed_column).isin(uppercase_sentinels), lit(None)).otherwise(trimmed_column)

dq_execution_log: list[Row] = []

def execute_data_quality_check(
    table_name: str,
    check_name: str,
    target_dataframe: DataFrame,
    valid_condition: Column
) -> bool:
    """Avalia conformidade de uma condição lógica no DataFrame e registra o resultado no log de qualidade."""
    total_records = target_dataframe.count()
    failed_records = target_dataframe.filter(~valid_condition).count()
    check_passed = (failed_records == 0)

    dq_execution_log.append(
        Row(
            table_name=table_name,
            check_name=check_name,
            total_records=total_records,
            failed_records=failed_records,
            passed=check_passed,
            checked_at=datetime.now(ZoneInfo("America/Recife"))
        )
    )
    status_tag = "PASS" if check_passed else "FAIL"
    print(f"[{status_tag}] {table_name} | {check_name}: {failed_records}/{total_records} falhas.")
    return check_passed

def execute_uniqueness_check(
    table_name: str,
    check_name: str,
    target_dataframe: DataFrame,
    key_columns: list[str]
) -> bool:
    """Valida unicidade das chaves especificadas no DataFrame e registra o resultado no log de qualidade."""
    total_records = target_dataframe.count()
    duplicate_keys = (
        target_dataframe
        .groupBy(*key_columns)
        .count()
        .filter(col("count") > 1)
        .count()
    )
    check_passed = (duplicate_keys == 0)

    dq_execution_log.append(
        Row(
            table_name=table_name,
            check_name=check_name,
            total_records=total_records,
            failed_records=duplicate_keys,
            passed=check_passed,
            checked_at=datetime.now(ZoneInfo("America/Recife"))
        )
    )
    status_tag = "PASS" if check_passed else "FAIL"
    print(f"[{status_tag}] {table_name} | {check_name}: {duplicate_keys} duplicatas em {total_records} registros.")
    return check_passed

def persist_data_quality_log() -> None:
    """Consolida os testes de qualidade executados e persiste na tabela Delta de auditoria."""
    if dq_execution_log:
        dq_dataframe = spark.createDataFrame(dq_execution_log)
        write_dataframe(
            dataframe=dq_dataframe,
            table_name="silver.tb_data_quality_log",
            save_mode="append"
        )
        print("Log de Data Quality persistido com sucesso na tabela silver.tb_data_quality_log")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/20 14:02:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. silver.tb_cotacao_dolar (Origem: bronze.tb_cotacao_dolar)

### Regras de Negócio:
- **Série Temporal Contínua**: Construção de calendário contínuo sem lacunas.
- **Forward Fill**: Dias sem cotação (finais de semana e feriados) herdam a última cotação útil disponível.

In [ ]:
EXCHANGE_RATE_PRECISION = "decimal(18,4)"

TbCotacaoDolarSilverSchema = StructType([
    StructField("data_cotacao", DateType(), nullable=False),
    StructField("cotacao_compra", DecimalType(18, 4), nullable=True),
    StructField("ingestion_datetime", TimestampType(), nullable=True),
])

def transform_tb_cotacao_dolar(raw_cotacao_dataframe: DataFrame) -> DataFrame:
    """Trata cotações da API PTAX, aplicando forward fill para fins de semana e feriados."""
    # Deduplicação diária: seleciona a cotação e ingestão mais recentes caso haja múltiplos registros no mesmo dia
    window_daily_quote = Window.partitionBy(to_date(col("dataHoraCotacao"))).orderBy(col("dataHoraCotacao").desc(), col("ingestion_datetime").desc())
    daily_quotes_dataframe = (
        raw_cotacao_dataframe
        .withColumn("data_cotacao", to_date(col("dataHoraCotacao")))
        .withColumn("_row_order_number", row_number().over(window_daily_quote))
        .filter(col("_row_order_number") == 1)
        .select(
            col("data_cotacao"),
            col("cotacaoCompra").cast(EXCHANGE_RATE_PRECISION).alias("cotacao_compra_bruta"),
            col("ingestion_datetime")
        )
    )

    # Como a API do Banco Central não possui cotações em finais de semana e feriados,
    # estruture o histórico de forma a garantir uma série temporal contínua,
    # gerando um calendário completo de datas entre a cotação mínima e máxima observadas.
    date_bounds = daily_quotes_dataframe.select(spark_min("data_cotacao"), spark_max("data_cotacao")).first()
    minimum_date, maximum_date = date_bounds[0], date_bounds[1]

    continuous_calendar = spark.sql(
        f"SELECT explode(sequence(to_date('{minimum_date}'), to_date('{maximum_date}'), interval 1 day)) as data_cotacao"
    )

    # Aplicando Forward Fill de modo que dias sem cotação
    # (finais de semana e feriados) recebam o valor do último dia útil disponível.
    forward_fill_window = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)

    transformed_dataframe = (
        continuous_calendar
        .join(daily_quotes_dataframe, on="data_cotacao", how="left")
        .withColumn("cotacao_compra", last("cotacao_compra_bruta", ignorenulls=True).over(forward_fill_window))
        .withColumn("ingestion_datetime", last("ingestion_datetime", ignorenulls=True).over(forward_fill_window))
        .select("data_cotacao", "cotacao_compra", "ingestion_datetime")
    )
    return enforce_dataframe_schema(transformed_dataframe, TbCotacaoDolarSilverSchema)

dataframe_cotacao_dolar_bronze = spark.table("bronze.tb_cotacao_dolar")
dataframe_cotacao_dolar_silver = transform_tb_cotacao_dolar(dataframe_cotacao_dolar_bronze)
write_dataframe(dataframe_cotacao_dolar_silver, table_name="silver.tb_cotacao_dolar")
dataframe_cotacao_dolar_silver.printSchema()
display(dataframe_cotacao_dolar_silver)

execute_uniqueness_check("silver.tb_cotacao_dolar", "unicidade_data_cotacao", dataframe_cotacao_dolar_silver, ["data_cotacao"])
execute_data_quality_check("silver.tb_cotacao_dolar", "cotacao_compra_positiva", dataframe_cotacao_dolar_silver, col("cotacao_compra") > 0)


root
 |-- data_cotacao: date (nullable = false)
 |-- cotacao_compra: decimal(18,4) (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



+------------+--------------+--------------------------+
|data_cotacao|cotacao_compra|ingestion_datetime        |
+------------+--------------+--------------------------+
|2026-09-14  |5.1690        |2026-09-19 23:55:35.966811|
|2026-09-15  |5.1484        |2026-09-19 23:55:35.966811|
|2026-09-16  |5.1520        |2026-09-19 23:55:35.966811|
|2026-09-17  |5.1515        |2026-09-19 23:55:35.966811|
|2026-09-18  |5.1569        |2026-09-19 23:55:35.966811|
+------------+--------------+--------------------------+



[PASS] silver.tb_cotacao_dolar | unicidade_data_cotacao: 0 duplicatas em 5 registros.


[PASS] silver.tb_cotacao_dolar | cotacao_compra_positiva: 0/5 falhas.


True

## 2. silver.tb_info_filmes (Origem: bronze.tb_movies_info)

### Regras de Negócio:
- **Deduplicação**: Unicidade por `id` mantendo o registro mais recente por `ingestion_datetime`.
- **Tradução e Limpeza de Status**: Normalização textual e mapeamento para português com fallback `'Não Informado'`.
- **Conversão Multi-Formato de Data**: Teste de múltiplos formatos aceitos e extração do `ano_lancamento`.

In [ ]:
DEFAULT_STATUS_FALLBACK = "Não Informado"

STATUS_TRANSLATION_MAP = {
    "RELEASED": "Lançado",
    "POST PRODUCTION": "Pós-Produção",
    "IN PRODUCTION": "Em Produção",
    "PLANNED": "Planejado",
    "RUMORED": "Rumores",
    "CANCELED": "Cancelado"
}

SUPPORTED_DATE_FORMATS = [
    "yyyy-MM-dd", "yyyy/MM/dd", "dd/MM/yyyy",
    "MM/dd/yyyy", "dd-MM-yyyy", "MM-dd-yyyy"
]

TbInfoFilmesSilverSchema = StructType([
    StructField("id_filme", IntegerType(), nullable=True),
    StructField("titulo", StringType(), nullable=True),
    StructField("titulo_original", StringType(), nullable=True),
    StructField("data_lancamento", DateType(), nullable=True),
    StructField("ano_lancamento", IntegerType(), nullable=True),
    StructField("duracao_minutos", IntegerType(), nullable=True),
    StructField("idioma_original", StringType(), nullable=True),
    StructField("status_filme", StringType(), nullable=True),
    StructField("sinopse", StringType(), nullable=True),
    StructField("frase_divulgacao", StringType(), nullable=True),
    StructField("ingestion_datetime", TimestampType(), nullable=True),
])

def clean_and_translate_status(column: Column) -> Column:
    """Normaliza e traduz para português o status de produção cinematográfica."""
    # Limpeza e Tradução do Status: Antes de traduzir o status_filme
    # (Released -> Lançado; Post Production -> Pós-Produção; In Production -> Em Produção;
    # Planned -> Planejado; Rumored -> Rumores; Canceled -> Cancelado), você deve normalizar a coluna.
    # A coluna de status deve ser normalizada (removendo ruídos, hífens sobressalentes e padronizando a caixa)
    # antes da tradução dos termos para o português.
    normalized_status = regexp_replace(regexp_replace(upper(trim(column)), r"-", " "), r"\s+", " ")
    translation_mapping_expressions = [
        lit(item) for key_value_pair in STATUS_TRANSLATION_MAP.items() for item in key_value_pair
    ]
    status_lookup_map = create_map(translation_mapping_expressions)
    # Registros corrompidos ou não mapeáveis devem ser padronizados como 'Não Informado'.
    return coalesce(status_lookup_map[normalized_status], lit(DEFAULT_STATUS_FALLBACK))

def parse_multiformat_date(column: Column) -> Column:
    """Converte sequencialmente múltiplos formatos de data textuais para DateType."""
    # Tratamento de Data Multi-Formato: Converta o campo de data de lançamento testando os diferentes
    # padrões presentes na origem de forma robusta. Apenas valores onde a conversão for estritamente
    # impossível devem ser tratados como ausentes (NULL).
    date_parsing_candidates = [try_to_date(column, single_date_format) for single_date_format in SUPPORTED_DATE_FORMATS]
    return coalesce(*date_parsing_candidates)

def transform_tb_movies_info(raw_movies_dataframe: DataFrame) -> DataFrame:
    """Padroniza metadados descritivos dos filmes (título, status, lançamento e classificação indicativa)."""
    # Deduplicação: A tabela deve conter unicidade por filme. Havendo registros duplicados na origem,
    # mantenha exclusivamente a versão mais recente com base na data de ingestão (ingestion_datetime).
    deduplicated = deduplicate_latest(raw_movies_dataframe, business_key_column="id")
    parsed_date_expression = parse_multiformat_date(col("release_date"))

    transformed_dataframe = deduplicated.select(
        col("id").cast("integer").alias("id_filme"),
        sanitize_string(col("title")).alias("titulo"),
        sanitize_string(col("original_title")).alias("titulo_original"),
        parsed_date_expression.alias("data_lancamento"),
        # Coluna Derivada: Criando a coluna ano_lancamento, extraída de data_lancamento.
        year(parsed_date_expression).alias("ano_lancamento"),
        col("runtime").cast("integer").alias("duracao_minutos"),
        sanitize_string(col("original_language")).alias("idioma_original"),
        clean_and_translate_status(col("status")).alias("status_filme"),
        sanitize_string(col("overview")).alias("sinopse"),
        sanitize_string(col("tagline")).alias("frase_divulgacao"),
        col("ingestion_datetime")
    )
    return enforce_dataframe_schema(transformed_dataframe, TbInfoFilmesSilverSchema)

dataframe_movies_info_bronze = spark.table("bronze.tb_movies_info")
dataframe_movies_info_silver = transform_tb_movies_info(dataframe_movies_info_bronze)
write_dataframe(dataframe_movies_info_silver, table_name="silver.tb_info_filmes")

display(dataframe_movies_info_silver)

execute_uniqueness_check("silver.tb_info_filmes", "unicidade_id_filme", dataframe_movies_info_silver, ["id_filme"])
execute_data_quality_check("silver.tb_info_filmes", "titulo_obrigatorio", dataframe_movies_info_silver, col("titulo").isNotNull())



[Stage 49:====================================================>   (16 + 1) / 17]



+--------+---------------------+---------------------+---------------+--------------+---------------+---------------+------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+--------------------------+
|id_filme|titulo               |titulo_original      |data_lancamento|ano_lancamento|duracao_minutos|idioma_original|status_filme|sinopse                                                                                                                                                                                                                                                       

[PASS] silver.tb_info_filmes | unicidade_id_filme: 0 duplicatas em 97879 registros.


[PASS] silver.tb_info_filmes | titulo_obrigatorio: 0/97879 falhas.


True

## 3. silver.tb_financeiro_filmes (Origem: bronze.tb_movies_financials)

### Regras de Negócio:
- **Higienização Monetária**: Extração de notações de escala (K, M, B), remoção de símbolos (`$`, `USD`) e conversão de valores $\le 0$ para `NULL`.
- **Deduplicação e Desempate por Maior Valor**: Decisão de arquitetura e negócio para assegurar unicidade por filme (`id`). A origem apresenta linhas duplicadas para o mesmo filme contendo valores conflitantes ou sentinelas nulos (ex.: linhas com valores válidos coexistindo com linhas zeradas/desconhecidas na mesma carga de ingestão). A fim de evitar perdas de métricas legítimas que ocorreriam com seleções arbitrárias ou funções determinísticas cegas (como hash de linha), optou-se expressamente por higienizar previamente os dados e priorizar o registro de maior valor preenchido (`receita_usd DESC NULLS LAST`, `orcamento_usd DESC NULLS LAST`), utilizando o `ingestion_datetime DESC` como desempate final.
- **Conversão Cambial**: Conversão para BRL via taxa PTAX da `silver.tb_cotacao_dolar`.
- **Métricas de Lucro e Margem**: Cálculo protegido de lucro (USD e BRL) e margem percentual segura.

In [ ]:
THOUSAND = 1_000
MILLION = 1_000_000
BILLION = 1_000_000_000
PERCENTAGE_FACTOR = 100
DECIMAL_PRECISION = "decimal(18,2)"
EXCHANGE_RATE_PRECISION = "decimal(18,4)"

TbFinanceiroFilmesSilverSchema = StructType([
    StructField("id_filme", IntegerType(), nullable=True),
    StructField("orcamento_usd", DecimalType(18, 2), nullable=True),
    StructField("receita_usd", DecimalType(18, 2), nullable=True),
    StructField("orcamento_brl", DecimalType(18, 2), nullable=True),
    StructField("receita_brl", DecimalType(18, 2), nullable=True),
    StructField("lucro_usd", DecimalType(18, 2), nullable=True),
    StructField("lucro_brl", DecimalType(18, 2), nullable=True),
    StructField("margem_lucro_percentual", DecimalType(18, 2), nullable=True),
    StructField("ingestion_datetime", TimestampType(), nullable=True),
])

def sanitize_currency_column(column: Column) -> Column:
    """Higieniza símbolos e sufixos monetários e converte valores para formato numérico decimal."""
    # Tratando valores textuais que representam ausência de dado (ex.: 'Unknown', 'Não Informado') como NULL antes da conversão de tipo.
    uppercase_sentinels = [sentinel.upper() for sentinel in DEFAULT_SENTINEL_VALUES]
    raw_column_text = upper(trim(column))
    # Higienizando as colunas de orçamento e receita para remover símbolos de moedas, pontuações de milhar e textos de ausência.
    column_with_replaced_sentinels = (
        when(raw_column_text.isin(uppercase_sentinels), lit(None))
        .otherwise(raw_column_text)
    )
    cleaned_without_symbols = regexp_replace(column_with_replaced_sentinels, r"[$\s]|USD", "")
    cleaned_without_punctuation = (regexp_replace(cleaned_without_symbols, r",", ""))

    # Tratamento de notações de escala abreviada (K para milhares, M para milhões, B para bilhões)
    extracted_thousands = regexp_extract(cleaned_without_punctuation, r"^([0-9.]+)[Kk]$", 1)
    extracted_millions = regexp_extract(cleaned_without_punctuation, r"^([0-9.]+)[Mm]$", 1)
    extracted_billions = regexp_extract(cleaned_without_punctuation, r"^([0-9.]+)[Bb]$", 1)
    extracted_plain = regexp_extract(cleaned_without_punctuation, r"^(-?[0-9.]+)$", 1)

    calculated_value = (
        when(extracted_thousands != "", extracted_thousands.cast("double") * THOUSAND)
        .when(extracted_millions != "", extracted_millions.cast("double") * MILLION)
        .when(extracted_billions != "", extracted_billions.cast("double") * BILLION)
        .when(extracted_plain != "", extracted_plain.cast("double"))
        .otherwise(lit(None))
    )
    # Converta as métricas para o tipo numérico decimal apropriado e garanta que valores zerados ou negativos sejam tratados como ausentes (NULL).
    return when(calculated_value > 0, calculated_value.cast(DECIMAL_PRECISION)).otherwise(lit(None))

def calculate_safe_percentage(
    numerator_column: Column,
    denominator_column: Column,
    scale_factor: int = PERCENTAGE_FACTOR,
    target_precision: str = DECIMAL_PRECISION
) -> Column:
    """Calcula percentual seguro entre numerador e denominador, prevenindo divisões por zero."""
    # Cálculo seguro de percentual evitando divisões por zero e propagando NULL para denominadores nulos ou inválidos
    valid_division =(
        numerator_column.isNotNull() 
        & denominator_column.isNotNull() 
        & (denominator_column > 0)
    )
    return when(valid_division, ((numerator_column / denominator_column) * scale_factor).cast(target_precision)).otherwise(lit(None))

def transform_tb_movies_financials(raw_financials_dataframe: DataFrame, exchange_rate_dollar_to_brl: float) -> DataFrame:
    """Converte orçamentos e receitas para USD e BRL, calculando lucro e margem operacional."""
    # Para garantir a preservação da melhor métrica financeira em empates de ingestão,
    # as colunas monetárias são higienizadas e tipadas previamente.
    budget_usd_expr = sanitize_currency_column(col("budget"))
    revenue_usd_expr = sanitize_currency_column(col("revenue"))

    cleaned_financials = (
        raw_financials_dataframe
        .withColumn("orcamento_usd", budget_usd_expr)
        .withColumn("receita_usd", revenue_usd_expr)
    )

    # Deduplicação: Como a base bruta possui linhas duplicadas com dados conflitantes ou nulos,
    # prioriza-se explicitamente o registro com os maiores valores preenchidos de receita e orçamento,
    # utilizando o timestamp de ingestão mais recente como critério final de desempate.
    window_financials_dedup = (
        Window
        .partitionBy("id")
        .orderBy(
            col("receita_usd").desc_nulls_last(),
            col("orcamento_usd").desc_nulls_last(),
            col("ingestion_datetime").desc()
        )
    )
    deduplicated = (
        cleaned_financials
        .withColumn("_row_order_number", row_number().over(window_financials_dedup))
        .filter(col("_row_order_number") == 1)
        .drop("_row_order_number")
    )
    rate_expr = lit(exchange_rate_dollar_to_brl).cast(EXCHANGE_RATE_PRECISION)

    col_orcamento_usd = col("orcamento_usd")
    col_receita_usd = col("receita_usd")
    budget_brl_expr = (col_orcamento_usd * rate_expr).cast(DECIMAL_PRECISION)
    revenue_brl_expr = (col_receita_usd * rate_expr).cast(DECIMAL_PRECISION)
    profit_usd_expr = (col_receita_usd - col_orcamento_usd).cast(DECIMAL_PRECISION)
    profit_brl_expr = (revenue_brl_expr - budget_brl_expr).cast(DECIMAL_PRECISION)

    transformed_dataframe = deduplicated.select(
        col("id").cast("integer").alias("id_filme"),
        col_orcamento_usd.alias("orcamento_usd"),
        col_receita_usd.alias("receita_usd"),
        budget_brl_expr.alias("orcamento_brl"),
        revenue_brl_expr.alias("receita_brl"),
        profit_usd_expr.alias("lucro_usd"),
        profit_brl_expr.alias("lucro_brl"),
        calculate_safe_percentage(profit_usd_expr, col_receita_usd).alias("margem_lucro_percentual"),
        col("ingestion_datetime")
    )
    return enforce_dataframe_schema(transformed_dataframe, TbFinanceiroFilmesSilverSchema)


# Leitura da cotação do dólar gerada na Silver para conversão cambial
dataframe_cotacao_silver = spark.table("silver.tb_cotacao_dolar")
latest_exchange_rate = (
    float(
        dataframe_cotacao_silver
        .orderBy(col("data_cotacao").desc())
        .select("cotacao_compra")
        .first()[0]
    )
)
print(f"Taxa de Cotação utilizada (PTAX Compra - Silver): R$ {latest_exchange_rate:.4f}")

dataframe_financials_bronze = spark.table("bronze.tb_movies_financials")
dataframe_financials_silver = transform_tb_movies_financials(dataframe_financials_bronze, latest_exchange_rate)
write_dataframe(dataframe_financials_silver, table_name="silver.tb_financeiro_filmes")
dataframe_financials_silver.printSchema()

display(dataframe_financials_silver)

# Avaliação das métricas de integridade lendo a tabela já persistida na Silver,
# truncando a árvore de execução complexa de expressões para evitar o estouro de 64 KB no Janino
dataframe_financials_persisted = spark.table("silver.tb_financeiro_filmes")
execute_uniqueness_check("silver.tb_financeiro_filmes", "unicidade_id_filme", dataframe_financials_persisted, ["id_filme"])

condition = col("lucro_usd").isNull() | (col("lucro_usd") == (col("receita_usd") - col("orcamento_usd")))
execute_data_quality_check("silver.tb_financeiro_filmes", "lucro_consistente", dataframe_financials_persisted, condition)


Taxa de Cotação utilizada (PTAX Compra - Silver): R$ 5.1569



[Stage 84:>                                                         (0 + 4) / 4]



root
 |-- id_filme: integer (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- orcamento_brl: decimal(18,2) (nullable = true)
 |-- receita_brl: decimal(18,2) (nullable = true)
 |-- lucro_usd: decimal(18,2) (nullable = true)
 |-- lucro_brl: decimal(18,2) (nullable = true)
 |-- margem_lucro_percentual: decimal(18,2) (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)




[Stage 87:>                                                         (0 + 4) / 4]



+--------+-------------+-----------+-------------+-----------+---------+---------+-----------------------+--------------------------+
|id_filme|orcamento_usd|receita_usd|orcamento_brl|receita_brl|lucro_usd|lucro_brl|margem_lucro_percentual|ingestion_datetime        |
+--------+-------------+-----------+-------------+-----------+---------+---------+-----------------------+--------------------------+
|1000011 |NULL         |NULL       |NULL         |NULL       |NULL     |NULL     |NULL                   |2026-09-19 23:55:33.697557|
|1000014 |NULL         |NULL       |NULL         |NULL       |NULL     |NULL     |NULL                   |2026-09-19 23:55:33.697557|
|1000030 |NULL         |NULL       |NULL         |NULL       |NULL     |NULL     |NULL                   |2026-09-19 23:55:33.697557|
|1000054 |NULL         |NULL       |NULL         |NULL       |NULL     |NULL     |NULL                   |2026-09-19 23:55:33.697557|
|1000073 |6000000.00   |NULL       |30960000.00  |NULL       |

[PASS] silver.tb_financeiro_filmes | unicidade_id_filme: 0 duplicatas em 99006 registros.
[PASS] silver.tb_financeiro_filmes | lucro_consistente: 0/99006 falhas.


True

## 4. silver.tb_metricas_engajamento (Origem: bronze.tb_movies_metrics)

### Regras de Negócio:
- **Higienização Numérica**: Substituição de vírgulas e casting seguro via `try_cast` contra column shift e dados textuais desalinhados.
- **Tratamento de Edge Case (Column Shift)**: Detecção de anomalias estruturais onde textos foram deslocados para colunas de notas/votos e anos de lançamento (ex.: 2018, 2019, 2020) foram parar indevidamente no campo de popularidade.
- **Validação de Limites de Negócio**: Notas médias no intervalo [0, 10] e contagens >= 0.


In [ ]:
MIN_RATING = 0.0
MAX_RATING = 10.0
MIN_VOTE_COUNT = 0

TbMetricasEngajamentoSilverSchema = StructType([
    StructField("id_filme", IntegerType(), nullable=True),
    StructField("popularidade", DoubleType(), nullable=True),
    StructField("nota_media_tmdb", DoubleType(), nullable=True),
    StructField("qtd_votos_tmdb", IntegerType(), nullable=True),
    StructField("nota_media_imdb", DoubleType(), nullable=True),
    StructField("qtd_votos_imdb", IntegerType(), nullable=True),
    StructField("ingestion_datetime", TimestampType(), nullable=True),
])

def sanitize_numeric_metric(
    column_name: str,
    target_data_type: str,
    minimum_value: float | None = None,
    maximum_value: float | None = None
) -> Column:
    """Limpa valores não numéricos, aplica cast e restringe métricas numéricas aos limites permitidos."""
    # Higienização de separador decimal e conversão segura protegida por try_cast para tolerar resíduos de texto sem interromper o pipeline.
    converted_numeric_expression = f"try_cast(regexp_replace(trim({column_name}), ',' , '.') as {target_data_type})"
    is_valid_metric_condition = expr(f"{converted_numeric_expression} IS NOT NULL")
    if minimum_value is not None:
        is_valid_metric_condition = is_valid_metric_condition & (expr(f"{converted_numeric_expression} >= {minimum_value}"))
    if maximum_value is not None:
        is_valid_metric_condition = is_valid_metric_condition & (expr(f"{converted_numeric_expression} <= {maximum_value}"))
    return when(is_valid_metric_condition, expr(converted_numeric_expression)).otherwise(lit(None))

def transform_tb_movies_metrics(raw_metrics_dataframe: DataFrame) -> DataFrame:
    """Higieniza métricas de popularidade, votos e durações dos filmes."""
    # Deduplicação: Unicidade por filme mantendo o registro mais recente por ingestion_datetime
    deduplicated = deduplicate_latest(raw_metrics_dataframe, business_key_column="id")

    # Regra de negócio anti-Column Shift: No arquivo bruto, linhas com deslocamento estrutural
    # projetaram textos descritivos e nomes em averageRating/numVotes, enquanto anos redondos (ex.: 2018, 2019, 2020)
    # foram parar na coluna de popularidade. Nesses casos, a métrica de popularidade é corrompida e deve ser convertida para NULL.
    corrupted_shift_condition = (
        col("averageRating").rlike("[a-zA-Z]")
        | col("numVotes").rlike("[a-zA-Z]")
        | col("popularity").rlike("[a-zA-Z]")
        | (trim(regexp_replace(col("popularity"), ",", ".")).rlike(r"^(18|19|20)[0-9]{2}$") & (expr("try_cast(vote_count as int)") < 30))
    )

    sanitized_popularity = when(
        corrupted_shift_condition, lit(None)
    ).otherwise(
        sanitize_numeric_metric("popularity", "double", minimum_value=0.0)
    )

    transformed_dataframe = deduplicated.select(
        col("id").cast("integer").alias("id_filme"),
        sanitized_popularity.alias("popularidade"),
        sanitize_numeric_metric("vote_average", "double", minimum_value=MIN_RATING, maximum_value=MAX_RATING).alias("nota_media_tmdb"),
        sanitize_numeric_metric("vote_count", "integer", minimum_value=MIN_VOTE_COUNT).alias("qtd_votos_tmdb"),
        sanitize_numeric_metric("averageRating", "double", minimum_value=MIN_RATING, maximum_value=MAX_RATING).alias("nota_media_imdb"),
        sanitize_numeric_metric("numVotes", "integer", minimum_value=MIN_VOTE_COUNT).alias("qtd_votos_imdb"),
        col("ingestion_datetime")
    )
    return enforce_dataframe_schema(transformed_dataframe, TbMetricasEngajamentoSilverSchema)

dataframe_metrics_bronze = spark.table("bronze.tb_movies_metrics")
dataframe_metrics_silver = transform_tb_movies_metrics(dataframe_metrics_bronze)
write_dataframe(dataframe_metrics_silver, table_name="silver.tb_metricas_engajamento")
dataframe_metrics_silver.printSchema()
display(dataframe_metrics_silver)

execute_uniqueness_check("silver.tb_metricas_engajamento", "unicidade_id_filme", dataframe_metrics_silver, ["id_filme"])
execute_data_quality_check("silver.tb_metricas_engajamento", "faixa_nota_tmdb", dataframe_metrics_silver, col("nota_media_tmdb").isNull() | col("nota_media_tmdb").between(0.0, 10.0))


root
 |-- id_filme: integer (nullable = true)
 |-- popularidade: double (nullable = true)
 |-- nota_media_tmdb: double (nullable = true)
 |-- qtd_votos_tmdb: integer (nullable = true)
 |-- nota_media_imdb: double (nullable = true)
 |-- qtd_votos_imdb: integer (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



+--------+------------+---------------+--------------+---------------+--------------+--------------------------+
|id_filme|popularidade|nota_media_tmdb|qtd_votos_tmdb|nota_media_imdb|qtd_votos_imdb|ingestion_datetime        |
+--------+------------+---------------+--------------+---------------+--------------+--------------------------+
|1000073 |13.212      |6.8            |15            |NULL           |236           |2026-09-19 23:55:34.178959|
|1000096 |0.71        |10.0           |1             |6.3            |695           |2026-09-19 23:55:34.178959|
|1000146 |1.347       |2.3            |3             |5.3            |43            |2026-09-19 23:55:34.178959|
|1000176 |1.075       |7.1            |5             |NULL           |277           |2026-09-19 23:55:34.178959|
|1000264 |1.118       |6.0            |NULL          |7.2            |66            |2026-09-19 23:55:34.178959|
+--------+------------+---------------+--------------+---------------+--------------+-----------

[PASS] silver.tb_metricas_engajamento | unicidade_id_filme: 0 duplicatas em 99013 registros.


[PASS] silver.tb_metricas_engajamento | faixa_nota_tmdb: 0/99013 falhas.


True

## 5. silver.tb_avaliacoes_usuarios (Origem: bronze.tb_movies_reviews)

### Regras de Negócio:
- **Deduplicação Integral**: Unicidade estrita por combinação completa `(id, nome, nota, comentario)`.
- **Validação de Escala**: Notas no intervalo [0, 10].
- **Fallback de Comentários**: Preenchimento de ausentes/em branco com `'Sem comentário'`.

In [6]:
MINIMUM_USER_RATING = 0.0
MAXIMUM_USER_RATING = 10.0
DEFAULT_COMMENT_FALLBACK = "Sem comentário"

TbAvaliacoesUsuariosSilverSchema = StructType([
    StructField("id_filme", IntegerType(), nullable=True),
    StructField("nome_usuario", StringType(), nullable=True),
    StructField("nota_usuario", DoubleType(), nullable=True),
    StructField("comentario_usuario", StringType(), nullable=True),
    StructField("ingestion_datetime", TimestampType(), nullable=True),
])

def transform_tb_movies_reviews(raw_reviews_dataframe: DataFrame) -> DataFrame:
    """Valida notas de usuários (escala 0-10) e preenche comentários ausentes com valor padrão."""
    # Remova registros integralmente duplicados (onde a combinação de filme, usuário, nota e comentário seja idêntica),
    # garantindo a unicidade das avaliações individuais.
    deduplicated = raw_reviews_dataframe.dropDuplicates(["id", "nome", "nota", "comentario"])
    trimmed_comment = trim(col("comentario"))
    valid_comment = trimmed_comment.isNotNull() & (trimmed_comment != "")
    parsed_rating = col("nota").cast("double")

    # Garanta que a nota atribuída pelo usuário respeite a escala permitida (0 a 10). Valores fora dessa faixa -> NULL.
    # Identifique comentários vazios ou compostos apenas por espaços e preencha com o texto padronizado 'Sem comentário'.
    transformed_dataframe = deduplicated.select(
        col("id").cast("integer").alias("id_filme"),
        trim(col("nome")).alias("nome_usuario"),
        when(parsed_rating.between(MINIMUM_USER_RATING, MAXIMUM_USER_RATING), parsed_rating).otherwise(lit(None)).alias("nota_usuario"),
        when(valid_comment, trimmed_comment).otherwise(lit(DEFAULT_COMMENT_FALLBACK)).alias("comentario_usuario"),
        col("ingestion_datetime")
    )
    return enforce_dataframe_schema(transformed_dataframe, TbAvaliacoesUsuariosSilverSchema)

dataframe_reviews_bronze = spark.table("bronze.tb_movies_reviews")
dataframe_reviews_silver = transform_tb_movies_reviews(dataframe_reviews_bronze)
write_dataframe(dataframe_reviews_silver, table_name="silver.tb_avaliacoes_usuarios")
dataframe_reviews_silver.printSchema()
display(dataframe_reviews_silver)

execute_data_quality_check("silver.tb_avaliacoes_usuarios", "faixa_nota_usuario", dataframe_reviews_silver, col("nota_usuario").isNull() | col("nota_usuario").between(0.0, 10.0))
execute_data_quality_check("silver.tb_avaliacoes_usuarios", "comentario_preenchido", dataframe_reviews_silver, col("comentario_usuario").isNotNull())


root
 |-- id_filme: integer (nullable = true)
 |-- nome_usuario: string (nullable = true)
 |-- nota_usuario: double (nullable = true)
 |-- comentario_usuario: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)

+--------+-------------------+------------+-------------------------------------------------+--------------------------+
|id_filme|nome_usuario       |nota_usuario|comentario_usuario                               |ingestion_datetime        |
+--------+-------------------+------------+-------------------------------------------------+--------------------------+
|797814  |Matheus Cavalcanti |1.1         |Péssimo em todos os sentidos.                    |2026-09-19 23:55:35.299475|
|613722  |Felipe Lopes 302   |8.5         |Obra-prima do cinema, simplesmente espetacular.  |2026-09-19 23:55:35.299475|
|924475  |Aline Barbosa 668  |9.6         |Adorei cada minuto, uma experiência inesquecível.|2026-09-19 23:55:35.299475|
|828887  |Lucas Fernandes 684|9.4   

[PASS] silver.tb_avaliacoes_usuarios | faixa_nota_usuario: 0/32412 falhas.


[PASS] silver.tb_avaliacoes_usuarios | comentario_preenchido: 0/32412 falhas.


True

## 6. silver.tb_generos (Origem: bronze.tb_credits_and_tags)

### Regras de Negócio:
- **Explosão Atômica**: Desmembramento por separadores múltiplos (`,`, `;`, `|`).
- **Filtragem de Column Shift**: Retenção estrita dos gêneros canônicos cinematográficos.

In [7]:
KNOWN_GENRES_LIST = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary",
    "Drama", "Family", "Fantasy", "History", "Horror", "Music",
    "Mystery", "Romance", "Science Fiction", "TV Movie", "Thriller",
    "War", "Western"
]

TbGenerosSilverSchema = StructType([
    StructField("id_filme", IntegerType(), nullable=True),
    StructField("nome_genero", StringType(), nullable=True),
    StructField("ingestion_datetime", TimestampType(), nullable=True),
])

def transform_tb_generos(raw_credits_dataframe: DataFrame) -> DataFrame:
    """Extrai e padroniza gêneros canônicos associados a cada filme a partir da lista original."""
    # Explodir (split + explode) a coluna genres, tratando a inconsistência de separadores (vírgula vs. ponto e vírgula vs. pipe) antes do split.
    normalized_genres = regexp_replace(col("genres"), r"[,;|]+", ",")
    sanitized_genre = trim(regexp_replace(col("_raw_genre"), r"""["'\[\]{}]""", ""))

    # Devido a falhas na estrutura de origem (Column Shift e separadores extras), remova resíduos em branco,
    # textos descritivos e valores numéricos deslocados que não pertençam ao domínio canônico de gêneros cinematográficos.
    transformed_dataframe = (
        raw_credits_dataframe
        .filter(col("genres").isNotNull())
        .withColumn("_raw_genre", explode(split(normalized_genres, ",")))
        .withColumn("nome_genero", sanitized_genre)
        .filter(col("nome_genero").isin(KNOWN_GENRES_LIST))
        .select(
            col("id").cast("integer").alias("id_filme"),
            col("nome_genero"),
            col("ingestion_datetime")
        )
        .dropDuplicates(["id_filme", "nome_genero"])
    )
    return enforce_dataframe_schema(transformed_dataframe, TbGenerosSilverSchema)

dataframe_credits_bronze = spark.table("bronze.tb_credits_and_tags")
dataframe_generos_silver = transform_tb_generos(dataframe_credits_bronze)
write_dataframe(dataframe_generos_silver, table_name="silver.tb_generos")
dataframe_generos_silver.printSchema()
display(dataframe_generos_silver)

execute_data_quality_check("silver.tb_generos", "genero_canonico_preenchido", dataframe_generos_silver, col("nome_genero").isNotNull())


root
 |-- id_filme: integer (nullable = true)
 |-- nome_genero: string (nullable = false)
 |-- ingestion_datetime: timestamp (nullable = true)



+--------+-----------+--------------------------+
|id_filme|nome_genero|ingestion_datetime        |
+--------+-----------+--------------------------+
|1036023 |Horror     |2026-09-19 23:55:34.748155|
|1051486 |History    |2026-09-19 23:55:34.748155|
|1096596 |Drama      |2026-09-19 23:55:34.748155|
|1101147 |Thriller   |2026-09-19 23:55:34.748155|
|1105684 |Comedy     |2026-09-19 23:55:34.748155|
|1195955 |Thriller   |2026-09-19 23:55:34.748155|
|1208017 |Mystery    |2026-09-19 23:55:34.748155|
|1218261 |Drama      |2026-09-19 23:55:34.748155|
|302946  |Crime      |2026-09-19 23:55:34.748155|
|347630  |Comedy     |2026-09-19 23:55:34.748155|
+--------+-----------+--------------------------+
only showing top 10 rows


[PASS] silver.tb_generos | genero_canonico_preenchido: 0/142149 falhas.


True

## 7. silver.tb_pessoas_empresas (Origem: bronze.tb_credits_and_tags)

### Regras de Negócio:
- **Dimensão Unificada**: Consolidação de `cast` (Ator), `directors` (Diretor), `writers` (Roteirista) e `production_companies` (Produtora).
- **Higienização e Padronização**: `initcap`, limpeza de caracteres especiais e eliminação de ruídos (URLs, imagens, números).

In [ ]:
# Dimensão unificada que consolida quatro tipos de entidade em uma única tabela:
# cast -> 'Ator'; directors -> 'Diretor'; writers -> 'Roteirista'; production_companies -> 'Produtora'
ENTITY_TYPE_MAPPINGS = [
    ("cast", "Ator"),
    ("directors", "Diretor"),
    ("writers", "Roteirista"),
    ("production_companies", "Produtora")
]

TbPessoasEmpresasSilverSchema = StructType([
    StructField("id_filme", IntegerType(), nullable=True),
    StructField("nome_entidade", StringType(), nullable=True),
    StructField("tipo_entidade", StringType(), nullable=True),
    StructField("ingestion_datetime", TimestampType(), nullable=True),
])

def extract_and_clean_entities(dataframe: DataFrame, source_column_name: str, entity_type_label: str) -> DataFrame:
    """Extrai entidades (pessoas ou produtoras) a partir de colunas concatenadas, limpando caracteres residuais."""
    # Desmembramento atômico por separadores múltiplos e padronização da capitalização (initcap)
    normalized_source = regexp_replace(col(source_column_name), r"[,;|]+", ",")
    sanitized_name = initcap(trim(regexp_replace(col("_raw_entity"), r"""["'\[\]{}]""", "")))

    # Expurgar resíduos de Column Shift: URLs, arquivos de imagem, valores puramente numéricos e sentinelas
    return (
        dataframe
        .filter(col(source_column_name).isNotNull())
        .withColumn("_raw_entity", explode(split(normalized_source, ",")))
        .withColumn("nome_entidade", sanitized_name)
        .withColumn("tipo_entidade", lit(entity_type_label))
        .filter(
            col("nome_entidade").isNotNull()
            & (col("nome_entidade") != "")
            & (~upper(col("nome_entidade")).isin(DEFAULT_SENTINEL_VALUES))
            & (~col("nome_entidade").rlike("^[0-9. -]+$"))
            & (~col("nome_entidade").rlike(r"(?i)\.(jpg|png|jpeg|webp)"))
            & (~col("nome_entidade").rlike(r"(?i)^(http|https|www\.)"))
            & (length(col("nome_entidade")) > 1)
        )
        .select(
            col("id").cast("integer").alias("id_filme"),
            col("nome_entidade"),
            col("tipo_entidade"),
            col("ingestion_datetime")
        )
    )

def transform_tb_pessoas_empresas(raw_credits_dataframe: DataFrame) -> DataFrame:
    """Unifica e tipifica pessoas (diretores/elenco) e produtoras vinculadas aos filmes."""
    # Deduplicação: Unicidade por filme mantendo o registro mais recente por ingestion_datetime
    # Consolidação dos quatro fluxos em modelo unificado categorizado pelo tipo de atuação
    extracted_dfs = [
        extract_and_clean_entities(raw_credits_dataframe, source_col, label)
        for source_col, label in ENTITY_TYPE_MAPPINGS
    ]
    unified_df = extracted_dfs[0]
    for additional_df in extracted_dfs[1:]:
        unified_df = unified_df.unionByName(additional_df)

    # Elimine registros duplicados pela tríade (id_filme, nome_entidade, tipo_entidade)
    transformed_dataframe = unified_df.dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
    return enforce_dataframe_schema(transformed_dataframe, TbPessoasEmpresasSilverSchema)

dataframe_pessoas_empresas_silver = transform_tb_pessoas_empresas(dataframe_credits_bronze)
write_dataframe(dataframe_pessoas_empresas_silver, table_name="silver.tb_pessoas_empresas")
dataframe_pessoas_empresas_silver.printSchema()
display(dataframe_pessoas_empresas_silver)

execute_data_quality_check("silver.tb_pessoas_empresas", "nome_entidade_obrigatorio", dataframe_pessoas_empresas_silver, col("nome_entidade").isNotNull())



[Stage 191:=====>                                                 (6 + 16) / 64]




[Stage 191:==========>                                           (12 + 16) / 64]




[Stage 191:===================>                                  (23 + 16) / 64]




[Stage 191:==========================>                           (31 + 16) / 64]




[Stage 191:===============================>                      (37 + 16) / 64]



...


[Stage 197:=============================>                        (35 + 16) / 64]




[Stage 197:==========================================>           (50 + 14) / 64]




[Stage 197:=================================================>     (58 + 6) / 64]



[PASS] silver.tb_pessoas_empresas | nome_entidade_obrigatorio: 0/896968 falhas.


True

## 8. Consolidação e Persistência do Log de Data Quality (Auditoria)

Persistência e visualização do histórico consolidado de auditoria de qualidade na camada Silver.

In [9]:
persist_data_quality_log()
dataframe_dq_log = spark.table("silver.tb_data_quality_log")
display(dataframe_dq_log.orderBy(col("checked_at").desc()))



[Stage 205:=====================================>                 (11 + 5) / 16]



Log de Data Quality persistido com sucesso em: /home/miguelsb/workspace/visagio/data/silver/tb_data_quality_log
+------------------------------+--------------------------+-------------+--------------+------+--------------------------+
|table_name                    |check_name                |total_records|failed_records|passed|checked_at                |
+------------------------------+--------------------------+-------------+--------------+------+--------------------------+
|silver.tb_pessoas_empresas    |nome_entidade_obrigatorio |896968       |0             |true  |2026-09-20 14:03:14.37158 |
|silver.tb_generos             |genero_canonico_preenchido|142149       |0             |true  |2026-09-20 14:02:56.823322|
|silver.tb_avaliacoes_usuarios |comentario_preenchido     |32412        |0             |true  |2026-09-20 14:02:54.851455|
|silver.tb_avaliacoes_usuarios |faixa_nota_usuario        |32412        |0             |true  |2026-09-20 14:02:54.563215|
|silver.tb_metricas_engajam